In [1]:
import pandas as pd

df = pd.read_excel("units/toat_platform.xlsx", skiprows=2)

In [2]:
folder_path = "wishlists/Annapurna Interactive Historical Revenues - releases_dates.csv"
dates  = pd.read_csv(folder_path)
dates = dates.dropna(subset='pc_release_date')
dates['pc_release_date'] = pd.to_datetime(dates['pc_release_date'])
release_date_dict = dates.groupby('product')['pc_release_date'].min().to_dict()

In [26]:
folder_path = "wishlists/Games Sales Curve Categorizations - overall.csv"
categories  = pd.read_csv(folder_path)


In [ ]:
release_date_dict.keys()

In [ ]:
TITLE =  'to a T'

In [ ]:
import glob
import os

# Define folder path
folder_path = "/Users/dougs/Documents/GitHub/IndieBI-Sales-EDA/units/"

# Get all .xlsx files in the folder
files = glob.glob(f"{folder_path}/*.csv")

# Read all files into DataFrames and add filename column
dfs = [pd.read_csv(file).assign(Source=os.path.basename(file)) for file in files]
av_df = pd.concat(dfs, ignore_index=True)
av_df['platform'] = av_df['platform'].str.title()
av_df['platform'] = av_df['platform'].replace({'Playstation': 'PlayStation'})
av_df['platform'] = av_df['platform'].replace({'Xbox': 'Microsoft'})


In [ ]:
curve_df = av_df.pivot(columns='platform', index='Weeks_from_Release', values="daily_delta").reset_index()

In [ ]:
platform_list = list(df.portal.unique())

In [ ]:
df = df.pivot(columns='portal', index='Day', values="Cumulative Selected Measure").reset_index()

In [ ]:
df['release_date'] = release_date_dict.get(TITLE)
df['release_date'] = pd.to_datetime(df['release_date'])

In [ ]:
df["DAR"] = (df["Day"] - df["release_date"]).dt.days
df["Weeks_from_Release"] = (df["DAR"] // 7) + 1  # Week 1 starts at 0-7 days


In [ ]:
#df = df[['Weeks_from_Release']+platform_list]

In [ ]:
actual_df = df.sort_values(by='DAR').drop_duplicates("Weeks_from_Release", keep='last')

In [ ]:
actual_df = actual_df.drop(["Day",'release_date'], axis=1)

In [ ]:
actual_df

In [ ]:
actual_df.to_clipboard()

In [ ]:
curve_df

In [ ]:
import pandas as pd
import numpy as np
actual_df = actual_df.set_index('Weeks_from_Release')
curve_df = curve_df.set_index('Weeks_from_Release')


# 2. Get the last actual week and values
last_week = actual_df.index.max()
last_values = actual_df.loc[last_week].copy()

# 3. Create a DataFrame to hold projections
projected_df = pd.DataFrame()

# 4. Iteratively apply the growth deltas week by week
current_values = last_values.copy()

for week in range(last_week + 1, curve_df.index.max() + 1):
    if week not in curve_df.index:
        continue
    deltas = curve_df.loc[week].fillna(0)  # Default to 0 growth if missing
    current_values = current_values * (1 + deltas)
    current_values.name = week
    current_values = np.ceil(current_values.fillna(0)).astype(int)
    projected_df = pd.concat([projected_df, current_values.to_frame().T])

# 5. Combine with actuals
full_df = pd.concat([actual_df, projected_df])

# 6. Reset index if needed
full_df = full_df.reset_index()

In [ ]:
full_df = full_df.drop("index", axis=1)

In [ ]:
filtered_df = full_df[full_df.index < 52]
filtered_df

In [ ]:
full_df.loc[51]

In [ ]:
full_df.loc[51].sum()

In [ ]:
full_df.loc[103].sum()

In [ ]:
full_df.loc[155].sum()

In [ ]:
curve_df.query("Weeks_from_Release <=52").to_clipboard()